# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset metadata and schema are provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset via the Croissant schema URL
dataset = mlc.Dataset(croissant_url)

# Print dataset level metadata
print(f"Dataset Title: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.datePublished}")

## 2. Data Overview
Examine available record sets with their `@id` values, fields, and columns. This helps identify the structure of the data.

> All entities are referenced by `@id` as per Croissant specification.

In [ ]:
# List all available record sets by `@id`
record_sets = list(dataset.metadata.recordSet)
for idx, rec_set in enumerate(record_sets):
    print(f"RecordSet {idx+1} @id: {rec_set['@id']}")
    print(f"  Name: {rec_set['name']}")
    print(f"  Description: {rec_set.get('description', '')}")
    # List all fields in this record set:
    print("  Fields:")
    for field in rec_set.get('field', []):
        print(f"    - @id: {field['@id']} | name: {field.get('name')} | type: {field.get('dataType')}")
        if 'column' in field and field['column']:
            print(f"      column(s): {[col['@id'] for col in (field['column'] if isinstance(field['column'],list) else [field['column']])]}")
    print("\n")
# Take note of which record set(s) and field(s) you want to analyze.

## 3. Data Extraction
Load data from record sets into DataFrames for analysis. We reference record sets and field/column names by their `@id`.

> _Below, select record sets by their actual `@id` from the previous step._

In [ ]:
# Prepare to extract all main record sets
# (Update this list based on your actual record set @ids discovered above)
# For illustration we will assume typical main @id. Replace as required with your dataset's @ids.
main_recordsets = [rs['@id'] for rs in dataset.metadata.recordSet]

dataframes = {}
for rec_id in main_recordsets:
    print(f"Loading records from: {rec_id}")
    records = list(dataset.records(record_set=rec_id))
    if records:
        dataframes[rec_id] = pd.DataFrame(records)
    else:
        print(f"No data loaded for {rec_id}")

# Choose the first loaded record set as example for EDA (update index if needed)
example_record_set = None
for k, df in dataframes.items():
    if not df.empty:
        example_record_set = k
        break

if example_record_set:
    print(f"\nRecord set DataFrame columns (@id):\n{dataframes[example_record_set].columns.tolist()}")
    display(dataframes[example_record_set].head())
else:
    print("No records found in any record set.")

## 4. Exploratory Data Analysis (EDA)
Conduct basic exploration: filter by a numeric field, normalize, and group if relevant.

> _Reference fields by their `@id` (column names) as shown above._

In [ ]:
if example_record_set is None:
    print('No data frame available for EDA.')
else:
    df = dataframes[example_record_set].copy()
    # Choose a numeric field @id; update as actual field in your record set
    # For demonstration, we attempt to auto-detect a plausible numeric field
    import numpy as np

    # Try to automatically select a numeric field
    numeric_field_id = None
    for col in df.columns:
        try:
            # coerce to numeric and check if there are at least 5 unique values
            coln = pd.to_numeric(df[col], errors='coerce')
            if coln.notnull().sum() > 5 and coln.nunique() > 3:
                numeric_field_id = col
                df[col] = coln
                break
        except Exception:
            continue
    if not numeric_field_id:
        print('No numeric field detected for analysis.')
    else:
        print(f'Example numeric field selected: {numeric_field_id}')
        # Filter - e.g., keep only records above a threshold (e.g., mean)
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}: {len(filtered_df)} records\n")

        # Normalize the field (Z-score)
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized column: {numeric_field_id}\n")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a likely categorical field (with < 20 unique values)
        group_field = None
        for col in df.columns:
            if col == numeric_field_id:
                continue
            if df[col].nunique() < 20 and df[col].dtype == object:
                group_field = col
                break
        if group_field and group_field in filtered_df.columns:
            print(f"\nGrouping by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame('mean_'+numeric_field_id)
            display(grouped_df.head())
        else:
            print('No suitable grouping field found.')

## 5. Visualization
Visualize data distributions for the selected numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if example_record_set is None or not numeric_field_id:
    print("No numeric field for visualization.")
else:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field and group_field in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=df, x=group_field, y=numeric_field_id)
        plt.title(f'{numeric_field_id} grouped by {group_field}')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook led you through loading the FAIR² mass spectrometry dataset via Croissant, exploring its structure using `@id` references, extracting data, filtering and transforming columns, and visualizing patterns in protein data. Adapt the variables and fields for further analysis as needed for your research application.